In [0]:
%sql
-- =============================================================================
-- ICEBASE — PHASE 2 | NOTEBOOK 06
-- Post-Pipeline Validation Queries
-- Idaho Mashers Hockey Analytics Platform
-- =============================================================================
-- THIS IS A STANDALONE VALIDATION NOTEBOOK — NOT A PIPELINE SOURCE FILE.
-- DO NOT add this to the pipeline. Run it manually after the pipeline
-- completes, in a regular notebook attached to the icebase-dev cluster
-- or via the Databricks SQL Editor.
--
-- PURPOSE:
--   Confirm all Silver tables exist with expected row counts and that
--   the Idaho Mashers narrative survived Bronze → Silver intact.
--
-- HOW TO USE:
--   1. Wait for the icebase-bronze-to-silver pipeline to show "Completed"
--   2. Open this notebook in Databricks, connect to icebase-dev cluster
--   3. Run each cell individually (Shift+Enter) and review outputs
--   4. All checks should match the expected results documented below
-- =============================================================================

-- COMMAND ----------
-- ── CHECK 1: Silver Table Row Counts ──────────────────────────────────────
-- Confirms all 5 Silver tables exist and have expected row counts.
-- Run this first — if any table is missing, the pipeline has an error.

SELECT 'dim_game'            AS table_name, COUNT(*) AS row_count FROM icebase.silver.dim_game
UNION ALL
SELECT 'dim_customer',                       COUNT(*)             FROM icebase.silver.dim_customer
UNION ALL
SELECT 'fact_tickets',                       COUNT(*)             FROM icebase.silver.fact_tickets
UNION ALL
SELECT 'bridge_promo',                       COUNT(*)             FROM icebase.silver.bridge_promo
UNION ALL
SELECT 'quarantine_tickets',                 COUNT(*)             FROM icebase.silver.quarantine_tickets
ORDER BY table_name;

-- EXPECTED RESULTS:
--   dim_game            82        (exactly 82 — one per game)
--   dim_customer        5,412+    (5,000 seed + 412 jersey night + any simulator drops)
--   fact_tickets        47,000+   (all seed transactions + simulator drops)
--   bridge_promo        4,000+    (redeemed promos from seed data)
--   quarantine_tickets  0         (no bad records in clean seed data)

-- COMMAND ----------
-- ── CHECK 2: Season Narrative Validation ──────────────────────────────────
-- Confirms the Idaho Mashers business story survived Bronze → Silver.
-- Look for 4 specific patterns documented below.

SELECT
  t.season_phase,
  COUNT(DISTINCT t.game_id)                                         AS games,
  COUNT(*)                                                          AS total_tickets,
  ROUND(AVG(t.ticket_price), 2)                                    AS avg_ticket_price,
  ROUND(
    SUM(CASE WHEN t.is_promo_ticket THEN 1 ELSE 0 END)
    * 100.0 / COUNT(*), 1
  )                                                                 AS promo_rate_pct,
  COUNT(DISTINCT t.customer_id)                                     AS unique_buyers,
  ROUND(SUM(t.ticket_price), 2)                                    AS total_revenue,
  ROUND(AVG(g.arena_fill_pct) * 100, 1)                           AS avg_fill_pct
FROM icebase.silver.fact_tickets  t
JOIN icebase.silver.dim_game      g ON t.game_id = g.game_id
GROUP BY t.season_phase
ORDER BY MIN(g.game_number);

-- EXPECTED PATTERNS — all 4 must be visible:
--   1. hot_start avg_fill_pct    ~91%    HIGHER than slump (~62%)
--   2. slump promo_rate_pct      ~41-45% HIGHEST of all phases
--   3. jersey_night avg_price    ~$130+  ROUGHLY 2x the season average (~$60)
--   4. hot_start promo_rate_pct  ~7-9%   LOWEST of all phases

-- COMMAND ----------
-- ── CHECK 3: Silver Cleaning Validation ───────────────────────────────────
-- Confirms the data quality work done in Silver is visible.
-- All dirty name records from Bronze should be clean here.

SELECT
  -- Whitespace cleaning check — should be 0
  COUNT(CASE WHEN full_name != TRIM(full_name)   THEN 1 END)  AS names_with_whitespace,
  -- ALL CAPS check — should be 0 (INITCAP fixed them)
  COUNT(CASE WHEN full_name = UPPER(full_name)
             AND LENGTH(full_name) > 2            THEN 1 END)  AS all_caps_names,
  -- Email normalization check — should be 0
  COUNT(CASE WHEN email != LOWER(email)           THEN 1 END)  AS mixed_case_emails,
  -- Tenure check — should all be positive
  COUNT(CASE WHEN tenure_days <= 0               THEN 1 END)  AS invalid_tenure,
  -- Null check — should be 0 (expect_or_drop removed them)
  COUNT(CASE WHEN customer_id IS NULL            THEN 1 END)  AS null_customer_ids
FROM icebase.silver.dim_customer;

-- EXPECTED: All columns = 0

-- COMMAND ----------
-- ── CHECK 4: Derived Column Validation ────────────────────────────────────
-- Confirms new Silver columns were computed correctly.

-- 4a: Seat tier rank mapping
SELECT
  section_tier,
  seat_tier_rank,
  COUNT(*) AS ticket_count
FROM icebase.silver.fact_tickets
GROUP BY section_tier, seat_tier_rank
ORDER BY seat_tier_rank DESC;

-- EXPECTED:
--   rinkside    5   (premium)
--   lower_bowl  4
--   mid_bowl    3
--   upper_bowl  2
--   standing    1   (cheapest)

-- COMMAND ----------
-- 4b: Advance purchase distribution
SELECT
  is_advance_purchase,
  COUNT(*)                          AS tickets,
  ROUND(AVG(ticket_price), 2)      AS avg_price,
  ROUND(COUNT(*) * 100.0
    / SUM(COUNT(*)) OVER (), 1)    AS pct_of_total
FROM icebase.silver.fact_tickets
GROUP BY is_advance_purchase;

-- EXPECTED:
--   TRUE  (advance buyers)  → higher avg_price, ~60-70% of total
--   FALSE (day-of buyers)   → lower avg_price, ~30-40% of total

-- COMMAND ----------
-- 4c: Playoff relevance flags in dim_game
SELECT
  season_phase,
  is_playoff_relevant,
  COUNT(*) AS games
FROM icebase.silver.dim_game
GROUP BY season_phase, is_playoff_relevant
ORDER BY MIN(game_number);

-- EXPECTED:
--   hot_start     FALSE   22 games
--   slump         FALSE   30 games
--   late_push     TRUE    20 games
--   jersey_night  TRUE     1 game
--   post_jersey   TRUE     5 games
--   bridge        FALSE    4 games (games 73-76)

-- COMMAND ----------
-- ── CHECK 5: Jersey Night Spotlight ───────────────────────────────────────
-- The most important single-game business story in the season.
-- Confirms the jersey night spike is visible and correctly attributed.

SELECT
  COUNT(*)                                                          AS total_tickets,
  SUM(CASE WHEN is_lapsed_reactivation  THEN 1 ELSE 0 END)        AS lapsed_reactivations,
  SUM(CASE WHEN is_jersey_night_game    THEN 1 ELSE 0 END)        AS jersey_night_tickets,
  ROUND(AVG(ticket_price), 2)                                      AS avg_ticket_price,
  ROUND(SUM(ticket_price), 2)                                      AS total_revenue,
  ROUND(
    SUM(CASE WHEN is_promo_ticket THEN 1 ELSE 0 END)
    * 100.0 / COUNT(*), 1
  )                                                                 AS promo_rate_pct
FROM icebase.silver.fact_tickets
WHERE is_jersey_night_game = TRUE;

-- EXPECTED:
--   total_tickets         6,200   (sellout — full arena capacity)
--   lapsed_reactivations  94      (deeply lapsed fans who came back)
--   avg_ticket_price      ~$130+  (2.4x price multiplier from phase weights)
--   promo_rate_pct        ~4%     (nearly no discounts — fans paid full price)

-- COMMAND ----------
-- ── CHECK 6: Promo Story Validation (The Bad Story) ───────────────────────
-- Confirms the promo over-reliance narrative is intact in bridge_promo.

SELECT
  season_phase,
  COUNT(*)                                  AS promo_count,
  ROUND(AVG(discount_pct) * 100, 1)        AS avg_discount_pct,
  ROUND(SUM(discount_amount), 2)            AS total_discount_given,
  ROUND(AVG(promo_impact_score), 2)         AS avg_impact_score,
  COUNT(CASE WHEN discount_tier = 'deep'
             THEN 1 END)                    AS deep_discounts
FROM icebase.silver.bridge_promo
GROUP BY season_phase
ORDER BY total_discount_given DESC;

-- EXPECTED:
--   slump phase has:
--     highest promo_count
--     highest total_discount_given
--     highest avg_discount_pct
--   This is the "bad story" — marketing eroded margins during the slump

-- COMMAND ----------
-- ── FINAL SUMMARY ─────────────────────────────────────────────────────────
-- Quick scorecard. Run last. All values should match expected column.

SELECT
  'dim_game rows = 82'                        AS check_name,
  CASE WHEN COUNT(*) = 82 THEN '✅ PASS' ELSE '❌ FAIL' END AS result
FROM icebase.silver.dim_game

UNION ALL SELECT
  'No dirty names in dim_customer',
  CASE WHEN COUNT(*) = 0  THEN '✅ PASS' ELSE '❌ FAIL' END
FROM icebase.silver.dim_customer
WHERE full_name != TRIM(full_name) OR full_name = UPPER(full_name)

UNION ALL SELECT
  'fact_tickets > 10k rows',
  CASE WHEN COUNT(*) > 10000 THEN '✅ PASS' ELSE '❌ FAIL' END
FROM icebase.silver.fact_tickets

UNION ALL SELECT
  'Quarantine table is empty',
  CASE WHEN COUNT(*) = 0  THEN '✅ PASS' ELSE '⚠️ REVIEW' END
FROM icebase.silver.quarantine_tickets

UNION ALL SELECT
  'Slump promo rate > Hot Start promo rate',
  CASE WHEN (
    SELECT ROUND(SUM(CASE WHEN is_promo_ticket THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1)
    FROM icebase.silver.fact_tickets WHERE season_phase = 'slump'
  ) > (
    SELECT ROUND(SUM(CASE WHEN is_promo_ticket THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1)
    FROM icebase.silver.fact_tickets WHERE season_phase = 'hot_start'
  ) THEN '✅ PASS' ELSE '❌ FAIL' END AS result;